In [10]:
import pandas as pd
import cupy as cp
import cudf
import cuml
import torch
import gc
from cuml.preprocessing import MaxAbsScaler
from cuml.preprocessing import MinMaxScaler
from cuml.preprocessing import Normalizer
from cuml.preprocessing import RobustScaler
from cuml.preprocessing import StandardScaler
from cuml.preprocessing import Binarizer
from cuml.preprocessing import FunctionTransformer
from cuml.preprocessing import KBinsDiscretizer
import time

In [11]:
df = cudf.read_csv('heart_disease_health_indicators_BRFSS2015.csv')
df

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0
253676,0.0,1.0,1.0,1.0,18.0,0.0,0.0,2.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0
253677,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0
253678,0.0,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0


In [12]:
df.columns

Index(['HeartDiseaseorAttack', 'HighBP', 'HighChol', 'CholCheck', 'BMI',
       'Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education',
       'Income'],
      dtype='object')

In [22]:
class NormalizationMultiClass(object):
    def __init__(self, dataset):
        global df_global
        global df_label_global
        self.dataset = dataset.copy().reset_index(drop = True)
        self.dataset_numpy_array = self.dataset.copy().to_numpy()
        self.X = cp.array(self.dataset_numpy_array)
        self.df_norm = self.dataset[['HeartDiseaseorAttack', 'HighBP', 'HighChol', 'CholCheck', 'BMI','Smoker', 'Stroke', 'Diabetes', 
                                     'PhysActivity', 'Fruits', 'Veggies',
                                     'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost','MentHlth', 'PhysHlth', 'DiffWalk', 
                                     'Sex', 'Age', 'Education','Income']]
        self.df_norm_numpy_array = self.df_norm.copy().to_numpy()
        self.dataset_label = self.dataset[['GenHlth']]
        self.df_global = self.df_norm
        self.df_label_global = self.dataset_label


    def MaxAbsScaler(self):
        global maxAbsScaler_global

        transformer = MaxAbsScaler().fit(self.df_norm_numpy_array)
        df_maxAbs_scaler_transform = transformer.transform(self.df_norm_numpy_array)
        
        maxAbsScaler_global = cudf.DataFrame(df_maxAbs_scaler_transform, columns = ['HeartDiseaseorAttack', 'HighBP', 'HighChol', 'CholCheck', 'BMI','Smoker', 'Stroke', 'Diabetes', 
                                     'PhysActivity', 'Fruits', 'Veggies',
                                     'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost','MentHlth', 'PhysHlth', 'DiffWalk', 
                                     'Sex', 'Age', 'Education','Income'])
        
    def MinMaxScaler(self):
        global MinMaxScaler_global

        transformer = MinMaxScaler().fit(self.df_norm_numpy_array)
        df_MinMaxScaler_transform = transformer.transform(self.df_norm_numpy_array)

        MinMaxScaler_global = cudf.DataFrame(df_MinMaxScaler_transform, columns = ['HeartDiseaseorAttack', 'HighBP', 'HighChol', 'CholCheck', 'BMI','Smoker', 'Stroke', 'Diabetes', 
                                     'PhysActivity', 'Fruits', 'Veggies',
                                     'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost','MentHlth', 'PhysHlth', 'DiffWalk', 
                                     'Sex', 'Age', 'Education','Income'])

    def Normalizer(self):
        global normalizer_global

        transformer = Normalizer().fit(self.df_norm_numpy_array)
        df_normalizer_transform = transformer.transform(self.df_norm_numpy_array)

        normalizer_global = cudf.DataFrame(df_normalizer_transform, columns = ['HeartDiseaseorAttack', 'HighBP', 'HighChol', 'CholCheck', 'BMI','Smoker', 'Stroke', 'Diabetes', 
                                     'PhysActivity', 'Fruits', 'Veggies',
                                     'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost','MentHlth', 'PhysHlth', 'DiffWalk', 
                                     'Sex', 'Age', 'Education','Income'])

    def RobustScaler(self):
        global robust_scaler_global

        transformer = RobustScaler().fit(self.df_norm_numpy_array)
        df_robust_scaler_transform = transformer.transform(self.df_norm_numpy_array)

        robust_scaler_global = cudf.DataFrame(df_robust_scaler_transform, columns = ['HeartDiseaseorAttack', 'HighBP', 'HighChol', 'CholCheck', 'BMI','Smoker', 'Stroke', 'Diabetes', 
                                     'PhysActivity', 'Fruits', 'Veggies',
                                     'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost','MentHlth', 'PhysHlth', 'DiffWalk', 
                                     'Sex', 'Age', 'Education','Income'])

    def StandardScaler(self):
        global standard_scaler_global

        transformer = StandardScaler().fit(self.df_norm_numpy_array)
        df_standard_scaler_transform = transformer.transform(self.df_norm_numpy_array)

        standard_scaler_global = cudf.DataFrame(df_standard_scaler_transform, columns = ['HeartDiseaseorAttack', 'HighBP', 'HighChol', 'CholCheck', 'BMI','Smoker', 'Stroke', 'Diabetes', 
                                     'PhysActivity', 'Fruits', 'Veggies',
                                     'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost','MentHlth', 'PhysHlth', 'DiffWalk', 
                                     'Sex', 'Age', 'Education','Income'])

    def Binarizer(self):
        global binarizer_global

        transformer = Binarizer().fit(self.df_norm_numpy_array)
        df_binarizer_transform = transformer.transform(self.df_norm_numpy_array)

        binarizer_global = cudf.DataFrame(df_binarizer_transform, columns = ['HeartDiseaseorAttack', 'HighBP', 'HighChol', 'CholCheck', 'BMI','Smoker', 'Stroke', 'Diabetes', 
                                     'PhysActivity', 'Fruits', 'Veggies',
                                     'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost','MentHlth', 'PhysHlth', 'DiffWalk', 
                                     'Sex', 'Age', 'Education','Income'])

    def FunctionTransformer(self):
        global function_transformer_global

        transformer = FunctionTransformer(func=cp.log1p)
        df_function_transformer_transform = transformer.transform(self.df_norm_numpy_array)

        function_transformer_global = cudf.DataFrame(df_function_transformer_transform, columns = ['HeartDiseaseorAttack', 'HighBP', 'HighChol', 'CholCheck', 'BMI','Smoker', 'Stroke', 'Diabetes', 
                                     'PhysActivity', 'Fruits', 'Veggies',
                                     'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost','MentHlth', 'PhysHlth', 'DiffWalk', 
                                     'Sex', 'Age', 'Education','Income'])

    def KBinsDiscretizer(self):
        global KBinsDiscretizer_global

        transformer = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform').fit(self.df_norm_numpy_array)
        df_KBinsDiscretizer_transform = transformer.transform(self.df_norm_numpy_array)

        KBinsDiscretizer_global = cudf.DataFrame(df_KBinsDiscretizer_transform, columns = ['HeartDiseaseorAttack', 'HighBP', 'HighChol', 'CholCheck', 'BMI','Smoker', 'Stroke', 'Diabetes', 
                                     'PhysActivity', 'Fruits', 'Veggies',
                                     'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost','MentHlth', 'PhysHlth', 'DiffWalk', 
                                     'Sex', 'Age', 'Education','Income'])

    def df_merge(self):
        global maxAbsScaler_merged_multiclass_global
        global MinMaxScaler_merged_multiclass_global
        global normalizer_merged_multiclass_global
        global robust_scaler_merged_multiclass_global
        global standard_scaler_merged_multiclass_global
        global binarizer_merged_multiclass_global
        global function_transformer_merged_multiclass_global
        global KBinsDiscretizer_merged__multiclass_global

        maxAbsScaler_merged_multiclass_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)
        MinMaxScaler_merged_multiclass_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)
        normalizer_merged_multiclass_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)
        robust_scaler_merged_multiclass_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)
        standard_scaler_merged_multiclass_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)
        binarizer_merged_multiclass_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)
        function_transformer_merged_multiclass_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)
        KBinsDiscretizer_merged__multiclass_global = cudf.concat([maxAbsScaler_global, self.df_label_global], axis = 1, ignore_index = False, sort = False)

        


    def main(self):
        st = time.time()
        self.MaxAbsScaler()
        self.MinMaxScaler()
        self.Normalizer()
        self.RobustScaler()
        self.StandardScaler()
        self.Binarizer()
        self.FunctionTransformer()
        self.KBinsDiscretizer()
        self.df_merge()

        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')

In [23]:
norm_multiclass = NormalizationMultiClass(df)

In [24]:
norm_multiclass.main()

Execution time: 0.494213342666626 seconds


In [25]:
maxAbsScaler_merged_multiclass_global.columns

Index(['HeartDiseaseorAttack', 'HighBP', 'HighChol', 'CholCheck', 'BMI',
       'Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'MentHlth',
       'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income', 'GenHlth'],
      dtype='object')

In [26]:
maxAbsScaler_merged_multiclass_global

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,GenHlth
0,0.0,1.0,1.0,1.0,0.408163,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.6,0.500000,1.0,0.0,0.692308,0.666667,0.375,5.0
1,0.0,0.0,0.0,0.0,0.255102,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.000000,0.0,0.0,0.538462,1.000000,0.125,3.0
2,0.0,1.0,1.0,1.0,0.285714,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,1.000000,1.0,0.0,0.692308,0.666667,1.000,5.0
3,0.0,1.0,0.0,1.0,0.275510,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.000000,0.0,0.0,0.846154,0.500000,0.750,2.0
4,0.0,1.0,1.0,1.0,0.244898,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.1,0.000000,0.0,0.0,0.846154,0.833333,0.500,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,0.459184,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.166667,0.0,1.0,0.384615,1.000000,0.875,3.0
253676,0.0,1.0,1.0,1.0,0.183673,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.000000,1.0,0.0,0.846154,0.333333,0.500,4.0
253677,0.0,0.0,0.0,1.0,0.285714,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.000000,0.0,0.0,0.153846,0.833333,0.250,1.0
253678,0.0,1.0,0.0,1.0,0.234694,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.000000,0.0,1.0,0.538462,0.833333,0.125,3.0


In [27]:
MinMaxScaler_merged_multiclass_global

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,GenHlth
0,0.0,1.0,1.0,1.0,0.408163,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.6,0.500000,1.0,0.0,0.692308,0.666667,0.375,5.0
1,0.0,0.0,0.0,0.0,0.255102,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.000000,0.0,0.0,0.538462,1.000000,0.125,3.0
2,0.0,1.0,1.0,1.0,0.285714,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,1.000000,1.0,0.0,0.692308,0.666667,1.000,5.0
3,0.0,1.0,0.0,1.0,0.275510,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.000000,0.0,0.0,0.846154,0.500000,0.750,2.0
4,0.0,1.0,1.0,1.0,0.244898,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.1,0.000000,0.0,0.0,0.846154,0.833333,0.500,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,0.459184,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.166667,0.0,1.0,0.384615,1.000000,0.875,3.0
253676,0.0,1.0,1.0,1.0,0.183673,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.000000,1.0,0.0,0.846154,0.333333,0.500,4.0
253677,0.0,0.0,0.0,1.0,0.285714,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.000000,0.0,0.0,0.153846,0.833333,0.250,1.0
253678,0.0,1.0,0.0,1.0,0.234694,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.000000,0.0,1.0,0.538462,0.833333,0.125,3.0
